# RectPack benchmark and summary statistics

Compare RectPack algorithms, sorting rules and rotation settings on
a shared collection of saved rectangle datasets.

Reusable functionality is imported from the `cutting_benchmark` package.
This notebook defines the experiment, inspects results and saves outputs.

The synthetic datasets are test scenarios, not representative
manufacturing data.

In [15]:
import sys
from itertools import product
from pathlib import Path

import pandas as pd

from cutting_benchmark.rectangle_datasets import load_datasets
from cutting_benchmark.solvers.rectpack_adapter import (
    ALGORITHMS,
    SORTS,
    run_pack,
)
from cutting_benchmark.benchmark import run_benchmark
from cutting_benchmark.summaries import summarise_results


In [16]:
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / ".git").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the repository root")

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "pilot_v1.json"
)

RESULTS_DIR = PROJECT_ROOT / "results" / "rectpack"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the saved datasets

Dataset generation is handled by the separate generation script.

This notebook loads the saved instances so that different solver
experiments can use the same input parts and sheet dimensions.

In [17]:
datasets = load_datasets(DATASET_PATH)

assert datasets, "No datasets were loaded"

dataset_ids = [
    dataset["dataset_id"]
    for dataset in datasets
]

assert len(set(dataset_ids)) == len(dataset_ids), (
    "Dataset IDs must be unique"
)

assert all(
    len(dataset["parts"]) == dataset["n_parts"]
    for dataset in datasets
), "Some datasets have inconsistent part counts"

dataset_overview = pd.DataFrame(
    [
        {
            "dataset_id": dataset["dataset_id"],
            "profile": dataset["profile"],
            "n_parts": dataset["n_parts"],
            "seed": dataset["seed"],
            "sheet_width": dataset["sheet_size"][0],
            "sheet_height": dataset["sheet_size"][1],
        }
        for dataset in datasets
    ]
)

print(f"Loaded {len(datasets)} datasets")
display(dataset_overview.head())

Loaded 30 datasets


,dataset_id,profile,n_parts,seed,sheet_width,sheet_height
0,rect_v1_mixed_sheet2500x1250_n35_seed0,mixed,35,0,2500,1250
1,rect_v1_mixed_sheet2500x1250_n35_seed1,mixed,35,1,2500,1250
2,rect_v1_mixed_sheet2500x1250_n35_seed2,mixed,35,2,2500,1250
3,rect_v1_mixed_sheet2500x1250_n35_seed3,mixed,35,3,2500,1250
4,rect_v1_mixed_sheet2500x1250_n35_seed4,mixed,35,4,2500,1250


In [18]:
workload_counts = (
    dataset_overview
    .groupby(["profile", "n_parts"], as_index=False)
    .agg(
        datasets=("dataset_id", "nunique"),
        seeds=("seed", "nunique"),
    )
)

display(workload_counts)

,profile,n_parts,datasets,seeds
0,large_and_small,35,5,5
1,large_and_small,100,5,5
2,mixed,35,5,5
3,mixed,100,5,5
4,strips,35,5,5
5,strips,100,5,5


## 2. Define the experiment

Compare all imported RectPack algorithms and sorting rules,
with rotation disabled and enabled.

Bin selection remains fixed at BBF inside the RectPack adapter.

Each configuration runs on every dataset. The benchmark function
shuffles execution order using a fixed execution seed.

In [19]:
CONFIGURATIONS = [
    {
        "algorithm": algorithm,
        "sort": sort,
        "rotation": rotation,
    }
    for algorithm, sort, rotation in product(
        ALGORITHMS,
        SORTS,
        [False, True],
    )
]

REPEATS = 1
AVAILABLE_SHEETS = None
EXECUTION_SEED = 123

expected_runs = (
    len(datasets)
    * len(CONFIGURATIONS)
    * REPEATS
)

print(f"Datasets: {len(datasets)}")
print(f"Configurations: {len(CONFIGURATIONS)}")
print(f"Repeats: {REPEATS}")
print(f"Expected benchmark runs: {expected_runs}")

display(pd.DataFrame(CONFIGURATIONS).head(8))

Datasets: 30
Configurations: 32
Repeats: 1
Expected benchmark runs: 960


,algorithm,sort,rotation
0,MaxRectsBssf,input_order,False
1,MaxRectsBssf,input_order,True
2,MaxRectsBssf,area,False
3,MaxRectsBssf,area,True
4,MaxRectsBssf,short_side,False
5,MaxRectsBssf,short_side,True
6,MaxRectsBssf,long_side,False
7,MaxRectsBssf,long_side,True


## 3. Check one solver run

Before running the full benchmark, confirm that a loaded dataset
can pass through the imported RectPack adapter.

In [20]:
example_dataset = datasets[0]

example_result = run_pack(
    parts=example_dataset["parts"],
    sheet_size=example_dataset["sheet_size"],
    algorithm="MaxRectsBssf",
    sort="area",
    rotation=False,
    available_sheets=AVAILABLE_SHEETS,
    return_layout=True,
)

assert example_result["solver"] == "rectpack"
assert example_result["complete"]
assert (
    example_result["parts_packed"]
    == example_dataset["n_parts"]
)

example_metrics = {
    key: value
    for key, value in example_result.items()
    if key not in {"placements", "used_sheets"}
}

display(pd.DataFrame([example_metrics]))

example_placements = pd.DataFrame(
    example_result["placements"],
    columns=[
        "bin_id",
        "x",
        "y",
        "width",
        "height",
        "part_id",
    ],
)

display(example_placements.head())

,solver,config_id,algorithm,sort,rotation,bin_selection,sheet_width,sheet_height,available_sheets,runtime_seconds,...,complete,individually_feasible,sheets_used,requested_area,packed_area,part_fraction,area_fraction,utilisation,area_lower_bound,sheet_gap_to_area_bound
0,rectpack,MaxRectsBssf__area__BBF,MaxRectsBssf,area,False,BBF,2500,1250,35,0.000975,...,True,True,5,12133094,12133094,1.0,1.0,0.776518,4,1


,bin_id,x,y,width,height,part_id
0,0,0,0,1240,601,P00018
1,0,1240,0,1247,597,P00001
2,0,0,601,1106,633,P00010
3,0,1106,601,1307,500,P00028
4,0,1106,1101,1061,141,P00000


## 4. Run the benchmark

Store one row per dataset–configuration–repeat combination.

Layouts are not included in the benchmark table. The single-run
example above demonstrates how to request a layout separately.

In [21]:
results = run_benchmark(
    datasets=datasets,
    configurations=CONFIGURATIONS,
    solver=run_pack,
    repeats=REPEATS,
    available_sheets=AVAILABLE_SHEETS,
    execution_seed=EXECUTION_SEED,
)

assert len(results) == expected_runs

run_keys = [
    "dataset_id",
    "solver",
    "config_id",
    "rotation",
    "repeat",
]

assert not results.duplicated(run_keys).any(), (
    "Duplicate benchmark runs found"
)

print(f"Finished {len(results)} benchmark runs")
print(f"Complete runs: {results['complete'].sum()}")
print(f"Incomplete runs: {(~results['complete']).sum()}")

display(results.head())

Finished 960 benchmark runs
Complete runs: 960
Incomplete runs: 0


,solver,config_id,algorithm,sort,rotation,bin_selection,sheet_width,sheet_height,available_sheets,runtime_seconds,...,area_fraction,utilisation,area_lower_bound,sheet_gap_to_area_bound,run_status,dataset_id,seed,profile,n_parts,repeat
0,rectpack,MaxRectsBaf__long_side__BBF,MaxRectsBaf,long_side,True,BBF,2500,1250,35,0.001643,...,1.0,0.768552,3,0,solution_returned,rect_v1_strips_sheet2500x1250_n35_seed2,2,strips,35,0
1,rectpack,MaxRectsBssf__area__BBF,MaxRectsBssf,area,False,BBF,2500,1250,100,0.006988,...,1.0,0.551820,20,16,solution_returned,rect_v1_large_and_small_sheet2500x1250_n100_seed3,3,large_and_small,100,0
2,rectpack,SkylineBl__long_side__BBF,SkylineBl,long_side,True,BBF,2500,1250,100,0.021736,...,1.0,0.748586,10,3,solution_returned,rect_v1_mixed_sheet2500x1250_n100_seed3,3,mixed,100,0
3,rectpack,MaxRectsBssf__input_order__BBF,MaxRectsBssf,input_order,True,BBF,2500,1250,100,0.003537,...,1.0,0.808841,10,2,solution_returned,rect_v1_mixed_sheet2500x1250_n100_seed4,4,mixed,100,0
4,rectpack,MaxRectsBaf__long_side__BBF,MaxRectsBaf,long_side,True,BBF,2500,1250,35,0.001212,...,1.0,0.797313,3,0,solution_returned,rect_v1_strips_sheet2500x1250_n35_seed0,0,strips,35,0


In [22]:
raw_results_path = (
    RESULTS_DIR / "rectpack_benchmark_raw.csv"
)

results.to_csv(
    raw_results_path,
    index=False,
)

print("Saved:", raw_results_path)

Saved: C:\Users\ivezs\2D-Optimisation-in-Metals-Manufacturing\results\rectpack\rectpack_benchmark_raw.csv


## 5. Inspect completion before comparing sheet counts

An incomplete packing must not be treated as a successful solution
using fewer sheets.

Inspect incomplete runs before calculating complete-order summaries.

In [23]:
incomplete_runs = results.loc[
    ~results["complete"]
].copy()

if incomplete_runs.empty:
    print("All runs packed the complete order.")
else:
    display(
        incomplete_runs[
            [
                "dataset_id",
                "algorithm",
                "sort",
                "rotation",
                "parts_total",
                "parts_packed",
                "sheets_used",
                "individually_feasible",
            ]
        ]
    )

    raise ValueError(
        "Inspect incomplete runs before continuing "
        "to the summary calculations."
    )

All runs packed the complete order.


## 6. Calculate summary tables

The summary function:

- Collapses repeated runs when their solution quality is identical.
- Uses median runtime across repeats.
- Compares configurations on matching datasets.
- Calculates best-observed sheet counts separately for each
  dataset and rotation setting.
- Produces overall and workload-specific summaries.

"Best observed" means best among the configurations supplied,
not necessarily the true optimum.

The default configuration identity is:
`solver`, `config_id` and `rotation`.

In [24]:
tables = summarise_results(results)

dataset_config = tables["dataset_config"]
completion_summary = tables["completion_summary"]
summary = tables["summary"]
summary_by_workload = tables["summary_by_workload"]

display(completion_summary)

,solver,config_id,rotation,datasets,completion_rate
0,rectpack,GuillotineBafSas__area__BBF,False,30,1.0
1,rectpack,GuillotineBafSas__area__BBF,True,30,1.0
2,rectpack,GuillotineBafSas__input_order__BBF,False,30,1.0
3,rectpack,GuillotineBafSas__input_order__BBF,True,30,1.0
4,rectpack,GuillotineBafSas__long_side__BBF,False,30,1.0
5,rectpack,GuillotineBafSas__long_side__BBF,True,30,1.0
6,rectpack,GuillotineBafSas__short_side__BBF,False,30,1.0
7,rectpack,GuillotineBafSas__short_side__BBF,True,30,1.0
8,rectpack,MaxRectsBaf__area__BBF,False,30,1.0
9,rectpack,MaxRectsBaf__area__BBF,True,30,1.0


### Interpretation of the main metrics

| Metric | Meaning |
|---|---|
| Mean sheets | Average sheet count across datasets |
| Mean utilisation | Average packed-area fraction of used sheets |
| Mean extra sheets | Average additional sheets versus the best observed configuration |
| Mean relative excess | Average proportional sheet overhead versus the best observed configuration |
| Tied-best rate | Fraction of datasets matching the fewest observed sheets; ties count for every matching configuration |
| Mean gap to area bound | Average sheet count above the area lower bound, not a proven optimality gap |
| Median runtime | Median of the per-dataset runtime values |
| P95 runtime | 95th percentile of the per-dataset runtime values |

The area lower bound is the total requested part area divided by the
sheet area, rounded upwards.

Geometry can prevent this bound from being achievable. A valid complete
packing that reaches it does establish the minimum sheet count for
that instance.

For complete orders on identical sheets, utilisation and sheet count
are linked; they are not independent measures of solution quality.

In [25]:
formats = {
    "mean_sheets": "{:.2f}",
    "mean_utilisation": "{:.1%}",
    "mean_extra_sheets": "{:.2f}",
    "mean_relative_excess": "{:.1%}",
    "tied_best_rate": "{:.1%}",
    "mean_gap_to_area_bound": "{:.2f}",
    "median_runtime": "{:.4f}",
    "p95_runtime": "{:.4f}",
}

summary_display = summary.copy()

for column, fmt in formats.items():
    summary_display[column] = (
        summary_display[column].map(fmt.format)
    )

display(summary_display)

,solver,config_id,rotation,datasets,mean_sheets,total_sheets,median_sheets,best_or_tied_datasets,mean_utilisation,mean_extra_sheets,mean_relative_excess,tied_best_rate,mean_gap_to_area_bound,median_runtime,p95_runtime
0,rectpack,MaxRectsBssf__long_side__BBF,False,30,13.80,414,10.0,26,68.7%,0.13,1.4%,86.7%,5.03,0.0020,0.0088
1,rectpack,MaxRectsBaf__long_side__BBF,False,30,13.87,416,10.0,24,68.0%,0.20,2.2%,80.0%,5.10,0.0023,0.0071
2,rectpack,GuillotineBafSas__long_side__BBF,False,30,13.87,416,10.0,24,67.7%,0.20,3.3%,80.0%,5.10,0.0016,0.0073
3,rectpack,MaxRectsBssf__area__BBF,False,30,13.90,417,10.0,24,66.7%,0.23,4.6%,80.0%,5.13,0.0030,0.0083
4,rectpack,MaxRectsBssf__short_side__BBF,False,30,13.93,418,10.0,22,66.5%,0.27,5.0%,73.3%,5.17,0.0023,0.0082
5,rectpack,MaxRectsBaf__area__BBF,False,30,13.90,417,10.0,23,66.4%,0.23,5.0%,76.7%,5.13,0.0024,0.0079
6,rectpack,MaxRectsBaf__short_side__BBF,False,30,13.93,418,10.0,22,66.2%,0.27,5.4%,73.3%,5.17,0.0023,0.0076
7,rectpack,SkylineBl__short_side__BBF,False,30,14.33,430,11.0,14,63.6%,0.67,9.2%,46.7%,5.57,0.0057,0.0259
8,rectpack,SkylineBl__long_side__BBF,False,30,14.53,436,10.5,8,62.7%,0.87,10.6%,26.7%,5.77,0.0074,0.0274
9,rectpack,MaxRectsBssf__input_order__BBF,False,30,14.57,437,11.0,10,62.0%,0.90,12.6%,33.3%,5.80,0.0021,0.0058


## 7. Examine performance by workload

Overall averages can hide differences between shape profiles
and problem sizes.

Inspect workload-specific results before deciding that one
configuration is consistently preferable.

In [26]:
workload_display = summary_by_workload.copy()

for column, fmt in formats.items():
    workload_display[column] = (
        workload_display[column].map(fmt.format)
    )

display(workload_display)

,profile,n_parts,solver,config_id,rotation,datasets,mean_sheets,total_sheets,median_sheets,best_or_tied_datasets,mean_utilisation,mean_extra_sheets,mean_relative_excess,tied_best_rate,mean_gap_to_area_bound,median_runtime,p95_runtime
0,large_and_small,35,rectpack,MaxRectsBssf__short_side__BBF,False,5,14.60,73,15.0,5,48.3%,0.00,0.0%,100.0%,7.00,0.0012,0.0015
1,large_and_small,35,rectpack,MaxRectsBssf__area__BBF,False,5,14.60,73,15.0,5,48.3%,0.00,0.0%,100.0%,7.00,0.0012,0.0026
2,large_and_small,35,rectpack,MaxRectsBaf__short_side__BBF,False,5,14.60,73,15.0,5,48.3%,0.00,0.0%,100.0%,7.00,0.0013,0.0016
3,large_and_small,35,rectpack,MaxRectsBaf__area__BBF,False,5,14.60,73,15.0,5,48.3%,0.00,0.0%,100.0%,7.00,0.0013,0.0016
4,large_and_small,35,rectpack,SkylineBl__short_side__BBF,False,5,14.60,73,15.0,5,48.3%,0.00,0.0%,100.0%,7.00,0.0037,0.0040
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,strips,100,rectpack,SkylineBl__short_side__BBF,True,5,8.80,44,9.0,0,72.2%,1.40,19.3%,0.0%,1.80,0.0129,0.0145
188,strips,100,rectpack,SkylineBl__input_order__BBF,True,5,9.20,46,9.0,0,69.1%,1.80,24.6%,0.0%,2.20,0.0111,0.0145
189,strips,100,rectpack,GuillotineBafSas__area__BBF,True,5,12.80,64,12.0,0,49.9%,5.40,73.2%,0.0%,5.80,0.0031,0.0032
190,strips,100,rectpack,GuillotineBafSas__input_order__BBF,True,5,13.40,67,13.0,0,47.7%,6.00,82.1%,0.0%,6.40,0.0030,0.0057


In [27]:
large_strips = summary_by_workload.query(
    "profile == 'strips' and n_parts == 100"
).copy()

for column, fmt in formats.items():
    large_strips[column] = (
        large_strips[column].map(fmt.format)
    )

display(large_strips)

,profile,n_parts,solver,config_id,rotation,datasets,mean_sheets,total_sheets,median_sheets,best_or_tied_datasets,mean_utilisation,mean_extra_sheets,mean_relative_excess,tied_best_rate,mean_gap_to_area_bound,median_runtime,p95_runtime
160,strips,100,rectpack,MaxRectsBssf__long_side__BBF,False,5,8.20,41,8.0,4,78.0%,0.20,2.5%,80.0%,1.20,0.0031,0.0034
161,strips,100,rectpack,GuillotineBafSas__long_side__BBF,False,5,8.20,41,8.0,4,77.6%,0.20,2.9%,80.0%,1.20,0.0020,0.0021
162,strips,100,rectpack,MaxRectsBssf__short_side__BBF,False,5,8.40,42,8.0,3,75.7%,0.40,5.4%,60.0%,1.40,0.0031,0.0054
163,strips,100,rectpack,MaxRectsBaf__long_side__BBF,False,5,8.40,42,8.0,3,75.8%,0.40,5.4%,60.0%,1.40,0.0036,0.0050
164,strips,100,rectpack,MaxRectsBssf__area__BBF,False,5,8.40,42,8.0,4,75.8%,0.40,5.7%,80.0%,1.40,0.0035,0.0036
165,strips,100,rectpack,MaxRectsBaf__short_side__BBF,False,5,8.60,43,9.0,2,74.0%,0.60,7.9%,40.0%,1.60,0.0033,0.0047
166,strips,100,rectpack,MaxRectsBaf__area__BBF,False,5,8.60,43,9.0,2,74.0%,0.60,7.9%,40.0%,1.60,0.0036,0.0054
167,strips,100,rectpack,SkylineBl__long_side__BBF,False,5,8.80,44,9.0,1,72.3%,0.80,10.4%,20.0%,1.80,0.0113,0.0138
168,strips,100,rectpack,SkylineBl__short_side__BBF,False,5,9.60,48,10.0,0,66.2%,1.60,20.8%,0.0%,2.60,0.0079,0.0124
169,strips,100,rectpack,MaxRectsBssf__input_order__BBF,False,5,9.60,48,10.0,1,66.3%,1.60,21.1%,20.0%,2.60,0.0031,0.0039


## 8. Save summary outputs

Save numeric tables rather than the formatted display copies.

- Raw results: individual benchmark runs.
- Dataset results: one row per dataset and configuration,
  after repeat aggregation.
- Completion summary: completion rates by configuration.
- Overall summary: metrics averaged across datasets.
- Workload summary: metrics grouped by profile and part count.

In [28]:
output_tables = {
    "rectpack_benchmark_dataset.csv": dataset_config,
    "rectpack_benchmark_completion.csv": completion_summary,
    "rectpack_benchmark_summary.csv": summary,
    "rectpack_summary_by_workload.csv": summary_by_workload,
}

for filename, table in output_tables.items():
    output_path = RESULTS_DIR / filename

    table.to_csv(
        output_path,
        index=False,
    )